# 🏙️ NYC 1940s Tax Photos - OCR + Ingest
### Powered by Elasticsearch Serverless + Mistral OCR

The NYC Municipal Archives hold ~720,000 photographs of **every building in the city**, shot 1939-1941 for tax assessment. Many show a signboard with a handwritten/stamped **block & lot** number - real-world material for OCR.

**What this notebook does:**
1. Loads scraped tax photos + metadata (run `tax_photos_scraper.py` first, or use the built-in demo seeds)
2. Runs each image through **Mistral OCR** (`mistral-ocr-latest`) to read the signboard / any text
3. Indexes the metadata + OCR text into `nyc_tax_photos` so you can search buildings by block, lot, address, or what the sign actually says
4. **(Optional)** adds Mistral embeddings for semantic search over the OCR text

> **Get the images first:** `python tax_photos_scraper.py --borough richmond --max 50`
> writes images + `metadata.csv` under `output/`. Or just run the demo cell below, which pulls a few photos directly by ID.

---
**Attribution for anything you publish:** *[Item Name], 1940s Tax Department photographs, Courtesy of the Municipal Archives, City of New York.* Non-commercial use is exempt from licensing.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import os, csv, glob, base64, requests
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless + set your Mistral key
> **Elastic Cloud Console → Your Project → Connection Details.** The Mistral key is handed out at the event.

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'
MISTRAL_API_KEY  = '<MISTRAL_API_KEY>'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Load the photos
**Path A - scraper output:** point `SCRAPE_DIR` at the scraper's `output/` folder (it has `metadata.csv` + images).
**Path B - demo seeds:** if no scraper output is found, the notebook downloads a few known photos directly so the notebook is self-contained.

In [ ]:
BASE = 'https://nycrecords.access.preservica.com'
UA = {'User-Agent': 'nyc-hacknight-taxphoto-ocr/1.0 (non-commercial research)'}
SCRAPE_DIR = 'output'   # where tax_photos_scraper.py wrote its files

items = []  # each: {io_uuid, borough, block, lot, title, start_date, end_date, image_bytes}

meta_csv = os.path.join(SCRAPE_DIR, 'metadata.csv')
if os.path.exists(meta_csv):
    with open(meta_csv) as f:
        for row in csv.DictReader(f):
            try:
                with open(row['file_path'], 'rb') as img:
                    row['image_bytes'] = img.read()
                items.append(row)
            except OSError:
                pass
    print(f'✅ Loaded {len(items)} photos from {SCRAPE_DIR}/')
else:
    # Demo seeds: a few confirmed Staten Island assets
    DEMO_IO = [
        'IO_039ddc71-0796-4a01-9043-aae5b1cf2080',
        'IO_02dbfda8-e39f-4254-b65f-23b8933201dd',
        'IO_13d2ff2f-ae36-4405-a02e-0eec3cea1db0',
    ]
    import re
    META_RE = re.compile(r'<span[^>]*>([A-Za-z ]+)</span><span class="metadata-field-separator">:</span>\s*<span class="metadata-content">(.*?)</span>', re.S)
    TAG_RE = re.compile(r'<[^>]+>')
    for io in DEMO_IO:
        html = requests.get(f'{BASE}/uncategorized/{io}/', headers=UA, timeout=30).text
        meta = {}
        for k, v in META_RE.findall(html):
            k = k.strip().lower().replace(' ', '_'); v = TAG_RE.sub('', v).strip()
            if v and k not in meta: meta[k] = v
        img = requests.get(f'{BASE}/download/file/{io}', headers=UA, timeout=30).content
        items.append({'io_uuid': io, 'borough': meta.get('borough', ''), 'block': meta.get('block', ''),
                      'lot': meta.get('lot', ''), 'title': meta.get('title', ''),
                      'start_date': meta.get('start_date', ''), 'end_date': meta.get('end_date', ''),
                      'image_bytes': img})
    print(f'✅ Downloaded {len(items)} demo photos')

## 3. OCR each photo with Mistral 🔎
Base64-encode the image and send it to the **Mistral OCR** endpoint. The response is markdown text extracted from the image.

> These are 1940s photos - some signboards are crisp, some barely legible. That's the point: it's a real OCR stress test. For tough handwriting, try a vision-capable **Mistral** chat model instead by prompting it to "read the signboard".

In [ ]:
def mistral_ocr(image_bytes):
    b64 = base64.b64encode(image_bytes).decode()
    r = requests.post('https://api.mistral.ai/v1/ocr',
        headers={'Authorization': f'Bearer {MISTRAL_API_KEY}', 'Content-Type': 'application/json'},
        json={'model': 'mistral-ocr-latest',
              'document': {'type': 'image_url', 'image_url': f'data:image/jpeg;base64,{b64}'}},
        timeout=60)
    r.raise_for_status()
    pages = r.json().get('pages', [])
    return '\n'.join(p.get('markdown', '') for p in pages).strip()

for it in items:
    try:
        it['ocr_text'] = mistral_ocr(it['image_bytes'])
    except Exception as e:
        it['ocr_text'] = ''
        print(f"  ! OCR failed for {it['io_uuid']}: {e}")
    print(f"  {it.get('title','?')[:40]:40s} → OCR: {it['ocr_text'][:50]!r}")
print('✅ OCR complete')

## 4. Create index and ingest

In [ ]:
INDEX = 'nyc_tax_photos'
if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
es.indices.create(index=INDEX, body={'mappings': {'properties': {
    'io_uuid':    {'type': 'keyword'},
    'borough':    {'type': 'keyword'},
    'block':      {'type': 'keyword'},
    'lot':        {'type': 'keyword'},
    'title':      {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'start_date': {'type': 'keyword'},
    'end_date':   {'type': 'keyword'},
    'ocr_text':   {'type': 'text'},
    'image_url':  {'type': 'keyword'},
}}})

def doc(it):
    return {'io_uuid': it['io_uuid'], 'borough': it.get('borough'), 'block': it.get('block'),
            'lot': it.get('lot'), 'title': it.get('title'), 'start_date': it.get('start_date'),
            'end_date': it.get('end_date'), 'ocr_text': it.get('ocr_text', ''),
            'image_url': f"{BASE}/download/file/{it['io_uuid']}"}

helpers.bulk(es, [{'_index': INDEX, '_source': doc(it)} for it in items], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {len(items)} photos into {INDEX}')

## 5. Verify

In [ ]:
resp = es.search(index=INDEX, size=0, body={'aggs': {'by_boro': {'terms': {'field': 'borough', 'size': 10}}}})
print(f"Total photos: {resp['hits']['total']['value']}")
for b in resp['aggregations']['by_boro']['buckets']:
    print(f"  {b['key']:30s} {b['doc_count']}")

# Find a specific property by block/lot
resp = es.search(index=INDEX, size=3, body={'query': {'bool': {'filter': [{'term': {'block': '1'}}]}}})
print('\nPhotos on block 1:')
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"  {s.get('title')} (lot {s.get('lot')}) — {s.get('image_url')}")

## 6. Going further with Mistral

Once the OCR text is indexed, add a `semantic_text` field backed by **Mistral embeddings** to search photos by what their signs say in natural language, or use a Mistral chat model to reason over the results. See [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md) for the `semantic_text` setup, or [mistral_guide.md](mistral_guide.md) to call Mistral directly.

---
## ✅ Index loaded
`nyc_tax_photos` holds your OCR'd 1940s photos. Explore it with queries like "show me buildings on block 2 in Staten Island" or "find photos whose signs mention a laundry".